# FaceGuard v2 — Presentation Attack Detection (PAD) MobileNetV3-Small Fine-Tuning

**Architecture:** `MobileNetV3-Small` (RGB Binary Classifier: `1.0` = Bona-fide / Genuine, `0.0` = Presentation Attack / Spoof)  
**Training Strategy:** Two-stage transfer learning with PyTorch DistributedDataParallel (DDP) across 2x T4 GPUs:  
- **Stage 0 (Sanity):** Overfit verification on a balanced mini-cohort before launching full training  
- **Stage 1 (Head-only):** Final linear layer (`classifier[-1]`) trained at `lr=1e-3` with frozen backbone  
- **Stage 2 (Finetune):** Last 3 inverted residual blocks (`features[-3:]`) + classifier at `lr=1e-4 / 1e-3` with frozen BatchNorm  
**Data Splitting:** Subject-disjoint train / validation / test manifests from official CelebA-Spoof JSON labels  
**Target Metrics:** Equal Error Rate (EER), APCER, BPCER, ACER pursuant to ISO/IEC 30107-3  
**Output Artifacts:** `mobilenetv3_pad.onnx` (dynamic batch, opset 17) & `pad_metrics.json`

---
### Kaggle Setup Instructions:
1. **Accelerator:** In the right sidebar, select **GPU T4 x2** (or **GPU P100**).
2. **Internet:** In the right sidebar, ensure **Internet On** is selected.
3. **Attach Inputs:** Click **`+ Add Input`**:
   - Attach **CelebA-Spoof** dataset (e.g. search `celeba spoof` or `marquis03/celeba-spoof-for-face-antispoofing`).
   - Attach **`faceguard-kaggle-source`** (the code bundle built via `scripts/build_kaggle_bundle.py`).

In [ ]:
# --- 1. Environment & Multi-GPU Verification ---
import os, sys, subprocess, shutil, json, zipfile
from pathlib import Path
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count()
print(f"Detected GPU Count: {gpu_count}")
for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {props.name} ({props.total_memory / (1024**3):.2f} GB VRAM)")

if gpu_count < 1:
    raise RuntimeError("CUDA GPU accelerator is required! Please select GPU T4 x2 in the Kaggle sidebar.")

# Install required runtime packages if missing
try:
    import onnx, onnxruntime
    print(f"[✓] ONNX Runtime: {onnxruntime.__version__}")
except ImportError:
    print("Installing onnx and onnxruntime...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)

WORK_ROOT = Path("/kaggle/working/faceguard")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"[✓] Working root initialized: {WORK_ROOT}")


In [ ]:
# --- 2. Locate and Unpack FaceGuard Code Bundle ---
import shutil, subprocess, sys, zipfile
from pathlib import Path

SOURCE_ROOT = None
target_src = WORK_ROOT / "source"

code_candidates = [
    Path("/kaggle/input/faceguard-kaggle-source"),
    Path("/kaggle/input/datasets/jit007/faceguard-kaggle-source"),
]

# 1. Fast direct check on explicit candidates
for cand in code_candidates:
    if cand.is_dir() and (cand / "src" / "faceguard").is_dir():
        SOURCE_ROOT = cand
        break
    if cand.is_file() and cand.suffix == ".zip":
        print(f"Unpacking source bundle from {cand}...")
        target_src.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(cand, "r") as zf:
            zf.extractall(target_src)
        SOURCE_ROOT = target_src
        break
    if cand.is_dir():
        zips = list(cand.glob("*.zip"))
        if zips:
            print(f"Unpacking source bundle from {zips[0]}...")
            target_src.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zips[0], "r") as zf:
                zf.extractall(target_src)
            SOURCE_ROOT = target_src
            break

# 2. Fast shallow scan of /kaggle/input (depth <= 2)
if SOURCE_ROOT is None and Path("/kaggle/input").exists():
    for ds in Path("/kaggle/input").iterdir():
        if not ds.is_dir():
            continue
        if (ds / "src" / "faceguard").is_dir():
            SOURCE_ROOT = ds
            break
        for f in ds.glob("*.zip"):
            if "faceguard" in f.name.lower() or "source" in f.name.lower():
                print(f"Unpacking source bundle from {f}...")
                target_src.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(f, "r") as zf:
                    zf.extractall(target_src)
                SOURCE_ROOT = target_src
                break
        if SOURCE_ROOT:
            break

# 3. Auto-fallback: If attached bundle is outdated or missing PAD files (pad_manifest.py, train_pad.py)
needs_clone = False
if SOURCE_ROOT is None:
    needs_clone = True
elif not (SOURCE_ROOT / "src" / "faceguard" / "pad_manifest.py").is_file():
    print(f"[!] Attached bundle at {SOURCE_ROOT} is an older version missing 'pad_manifest.py'.")
    needs_clone = True

if needs_clone:
    print("Fetching latest FaceGuard code directly from GitHub (includes PAD pipeline & manifests)...")
    git_target = WORK_ROOT / "repo"
    if git_target.exists():
        shutil.rmtree(git_target)
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/surjit10/face-recognition-liveness-detection.git",
        str(git_target)
    ], check=True)
    SOURCE_ROOT = git_target

# Clear any stale faceguard modules from sys.modules
for mod in list(sys.modules):
    if mod == "faceguard" or mod.startswith("faceguard."):
        del sys.modules[mod]

src_path = str((SOURCE_ROOT / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"[✓] Project source active from: {SOURCE_ROOT}")
from faceguard.pad_manifest import prepare_celeba_spoof_manifests
print("[✓] Verified: faceguard.pad_manifest loaded successfully!")


In [ ]:
# --- 3. Locate CelebA-Spoof Dataset Root ---
def find_celeba_root(start_path: Path) -> Path | None:
    def is_valid_root(p: Path) -> bool:
        return (p / "metas" / "intra_test" / "train_label.json").is_file() or \
               (p / "metas" / "labels" / "train_label.json").is_file()

    # 1. Candidate dataset root folders (including attentionlayer241 mount)
    base_candidates = [
        Path("/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing"),
        Path("/kaggle/input/celeba-spoof-for-face-antispoofing"),
        Path("/kaggle/input/celeba-spoof"),
        Path("/kaggle/input/celebaspoof"),
        Path("/kaggle/input/CelebA_Spoof"),
        start_path,
    ]

    # Auto-add any mounted datasets in /kaggle/input and /kaggle/input/datasets/*/*
    if start_path.exists():
        for p in start_path.iterdir():
            if p.is_dir() and p.name != "datasets":
                base_candidates.append(p)
        datasets_dir = start_path / "datasets"
        if datasets_dir.exists() and datasets_dir.is_dir():
            for u in datasets_dir.iterdir():
                if u.is_dir():
                    for ds in u.iterdir():
                        if ds.is_dir():
                            base_candidates.append(ds)

    # 2. Fast direct checks for CelebA_Spoof_ (with underscore) and other common structures
    for base in base_candidates:
        if not base.exists():
            continue
        sub_checks = [
            base / "CelebA_Spoof_",
            base / "CelebA_Spoof",
            base / "celeba_spoof",
            base,
            base / "CelebA_Spoof_" / "CelebA_Spoof",
            base / "CelebA_Spoof_" / "celeba_spoof",
            base / "CelebA_Spoof_" / "Data",
            base / "Data",
        ]
        for sub in sub_checks:
            if is_valid_root(sub):
                return sub

    # 3. Fast pruned search for train_label.json (skips Data/train/val/test image folders)
    for base in base_candidates:
        if not base.is_dir():
            continue
        bname = base.name.lower()
        if "celeba" not in bname and "spoof" not in bname:
            continue
        for root, dirs, files in os.walk(base):
            # Prune image and subject folders so we NEVER scan 2M images
            dirs[:] = [
                d for d in dirs
                if d.lower() not in {"data", "train", "val", "test", "images", "img", "imgs"}
                and not d.isdigit()
            ]
            if "train_label.json" in files:
                r = Path(root)
                if r.name in {"intra_test", "labels"} and r.parent.name == "metas":
                    return r.parent.parent
                if r.name == "metas":
                    return r.parent
                return r

    return None

CELEBA_ROOT = find_celeba_root(Path("/kaggle/input"))

# Diagnostic helper if not found
if CELEBA_ROOT is None:
    target = Path("/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing")
    if not target.exists():
        target = Path("/kaggle/input/celeba-spoof-for-face-antispoofing")
    print("\n[!] Diagnostic check of attached dataset contents:")
    if target.exists():
        print(f"Entries in {target}:")
        for item in list(target.iterdir())[:10]:
            print(f"  - {item.name} ({'dir' if item.is_dir() else 'file'})")
        sub_spoof = target / "CelebA_Spoof_"
        if sub_spoof.exists() and sub_spoof.is_dir():
            print(f"Entries in {sub_spoof}:")
            for item in list(sub_spoof.iterdir())[:10]:
                print(f"  - {item.name} ({'dir' if item.is_dir() else 'file'})")
    raise RuntimeError(
        "CelebA-Spoof dataset root with 'metas/.../train_label.json' could not be found!\n"
        "Check the diagnostic output printed above to verify where 'metas' is located."
    )

print(f"[✓] CELEBA_ROOT confirmed: {CELEBA_ROOT}")


In [ ]:
# --- 4. Prepare Official Subject-Disjoint PAD Manifests ---
from faceguard.pad_manifest import prepare_celeba_spoof_manifests

MANIFEST_DIR = WORK_ROOT / "pad_manifests"
if MANIFEST_DIR.exists():
    shutil.rmtree(MANIFEST_DIR)

print("Generating verified subject-disjoint train/val/test manifests from official JSON labels...")
manifest_report = prepare_celeba_spoof_manifests(
    dataset_root=CELEBA_ROOT,
    output_dir=MANIFEST_DIR,
    seed=42,
    validation_subject_fraction=0.10,
    train_limit=100_000,
    validation_limit=20_000,
    test_limit=40_000,
)

print("\n" + "="*60)
print("PAD MANIFEST SUMMARY (Subject-Disjoint Splits):")
for split_name in ["train", "val", "test"]:
    s = manifest_report["splits"][split_name]
    print(f"  • {split_name.upper():5s}: {s['samples']:,} samples across {s['subjects']:,} subjects | "
          f"Bona-fide (Live): {s['target_live_counts'].get(1, 0):,}, Attack (Spoof): {s['target_live_counts'].get(0, 0):,}")
print("="*60)


In [ ]:
# --- 5. Distributed 2-GPU MobileNetV3 Training via torchrun ---
PAD_OUTPUT = WORK_ROOT / "pad_output"
PAD_OUTPUT.mkdir(parents=True, exist_ok=True)

nproc = min(2, torch.cuda.device_count())
print(f"Launching PyTorch DDP on {nproc} GPU(s)...")

train_script = SOURCE_ROOT / "scripts" / "train_pad.py"

# Critical NCCL flags for Kaggle 2x Tesla T4 GPUs:
# Prevents NCCL deadlocking on virtualized PCIe P2P barriers & unbuffers live stdout
env = os.environ.copy()
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    sys.executable, "-u", "-m", "torch.distributed.run",
    "--standalone",
    f"--nproc_per_node={nproc}",
    str(train_script),
    "--dataset-root", str(CELEBA_ROOT),
    "--manifest-dir", str(MANIFEST_DIR),
    "--output-dir", str(PAD_OUTPUT),
    "--head-epochs", "2",
    "--finetune-epochs", "6",
    "--batch-size-per-gpu", "64",
    "--workers-per-gpu", "2",
    "--early-stopping-patience", "4",
    "--seed", "42",
    "--acknowledge-non-commercial-terms",
]

print(f"Running distributed training command:\n{' '.join(cmd)}\n")
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"PAD training failed with exit code {process.returncode}")
print("\n[✓] Training run completed successfully!")


In [ ]:
# --- 6. Verify Output Artifacts & Metrics ---
metrics_path = PAD_OUTPUT / "pad_metrics.json"
onnx_path = PAD_OUTPUT / "mobilenetv3_pad.onnx"
best_pth_path = PAD_OUTPUT / "mobilenetv3_pad_best.pth"

if not metrics_path.exists() or not onnx_path.exists():
    raise RuntimeError("Training output artifacts are missing!")

with metrics_path.open() as f:
    metrics = json.load(f)

val_m = metrics["validation_threshold"]
test_m = metrics["test"]

print("="*65)
print("🏆 PAD TEST SET EVALUATION (ISO/IEC 30107-3 Standard Metrics):")
print(f"  • Calibrated Threshold (τ* from Val EER): {val_m['threshold']:.4f}")
print(f"  • Validation EER:                         {val_m['acer']*100:.2f}%")
print(f"  • Test APCER (Spoof accepted as Live):    {test_m['apcer']*100:.2f}%")
print(f"  • Test BPCER (Live rejected as Spoof):    {test_m['bpcer']*100:.2f}%")
print(f"  • Test ACER (Average Classification Err): {test_m['acer']*100:.2f}%")
print(f"  • Test ROC-AUC:                           {test_m['auc']:.4f}")
print(f"  • ONNX Parity (Max Abs Diff):             {metrics['onnx_parity_max_abs_difference']:.2e}")
print("="*65)
print(f"[✓] Generated ONNX model: {onnx_path.name} ({onnx_path.stat().st_size / (1024*1024):.2f} MB)")
print(f"[✓] Best PyTorch Weights:  {best_pth_path.name} ({best_pth_path.stat().st_size / (1024*1024):.2f} MB)")


In [ ]:
# --- 7. Download Bundle Helper ---
from IPython.display import FileLink, display

bundle_path = Path("/kaggle/working/pad_model_bundle.zip")
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in [onnx_path, metrics_path, PAD_OUTPUT / "pad_config.json", best_pth_path]:
        if path.exists():
            zf.write(path, arcname=path.name)
            print(f"  [+] Bundled: {path.name}")

print(f"\n🎉 Download Bundle Ready: {bundle_path.name} ({bundle_path.stat().st_size / (1024*1024):.2f} MB)")
print("Click the link below to download your fine-tuned PAD model:")
display(FileLink("pad_model_bundle.zip"))
